# 02 Climate trends

Temporal and spatial trends in mean June-August SPEI-3.

| Output | Manuscript |
| --- | --- |
| `figures/figure_05_spei3_trends_1992_2024.pdf` | Figure 5 |
| `figures/figure_S01_spei3_trends_1950_2024.pdf` | Figure S1 |
| printed table | Table S6 source values |

Reads only `data/processed/`. No raw file is opened.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

from matplotlib.cm import ScalarMappable
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.ticker import FuncFormatter
from scipy import stats

from src import data_loading as dl

dl.set_plot_style()
dl.describe_paths()

spei_aimag = dl.load_spei("aimag_jja")
spei_national = dl.load_spei("national_jja")
spei_trends = dl.load_spei("aimag_trends")
aimags = dl.load_aimag_boundaries()

print(f"{len(spei_aimag)} aimag-year SPEI values, "
      f"{spei_aimag['Year'].min()}-{spei_aimag['Year'].max()}")

## Shared figure builder

Figures 5 and S3 differ only in their analysis period, so one function draws both.
Fonts are specified in final print points and multiplied by `FIG_SCALE`, which is the
supersampling factor. Every label therefore clears the AGU 8 pt floor at print size.

Aimag name labels are drawn at 8 pt rather than the 5 pt effective size used in the
submitted draft. If any label now collides, adjust `LABEL_OFFSETS` below.

In [ ]:
FIG_SCALE = 3.6

MAP_COLORS = ["#FFFFD4", "#FEE391", "#FEC44F", "#FE9929", "#D95F0E", "#993404"]
DIVERGING_COLORS = ["#2166AC", "#67A9CF", "#D1E5F0", "#FDDBC7", "#EF8A62", "#B2182B"]
TREND_COLOR = "#3F3F3F"

DISPLAY_NAMES = dl.WRAPPED_NAMES

LABEL_OFFSETS = {"Bayan-Ulgii": (-12, 6), "Khovd": (-6, -4), "Uvs": (2, 6),
                 "Zavkhan": (0, 4), "Govi-Altai": (-4, -6), "Arkhangai": (2, 2),
                 "Bayankhongor": (-2, -8), "Uvurkhangai": (6, -4),
                 "Khuvsgul": (0, 8), "Tuv": (-4, -14), "Dundgovi": (2, -2),
                 "Umnugovi": (0, -4), "Dornogovi": (6, -2),
                 "Sukhbaatar": (6, 2), "Khentii": (4, 4), "Dornod": (0, 2)}



def pt(points):
    return points * FIG_SCALE


def build_spei_trend_figure(start_year, end_year, tick_step):
    national = spei_national[
        spei_national["Year"].between(start_year, end_year)].copy()
    national_fit = stats.linregress(national["Year"], national["JJA_Mean_SPEI3"])
    national["OLS_fitted"] = (national_fit.intercept
                              + national_fit.slope * national["Year"])

    period = f"{start_year}-{end_year}"
    trends = spei_trends[spei_trends["Period"].eq(period)]
    if trends.empty:
        raise ValueError(f"No trends stored for period {period}")

    mapped = aimags.merge(trends, on="Aimag", how="left")
    if mapped["Drying_rate_per_decade"].isna().any():
        missing = mapped.loc[mapped["Drying_rate_per_decade"].isna(), "Aimag"].tolist()
        raise ValueError("Polygons without a trend value: " + ", ".join(missing))

    values = mapped["Drying_rate_per_decade"].to_numpy()
    if values.min() >= 0:
        cmap = ListedColormap(MAP_COLORS)
        boundaries = np.linspace(values.min(), values.max(), len(MAP_COLORS) + 1)
        colorbar_label = ("Rate of decline in mean June\u2013August monthly SPEI-3, "
                          f"{start_year}\u2013{end_year} "
                          "(SPEI units decade$^{-1}$)")
    else:
        cmap = ListedColormap(DIVERGING_COLORS)
        extreme = float(np.abs(values).max())
        boundaries = np.linspace(-extreme, extreme, len(DIVERGING_COLORS) + 1)
        colorbar_label = ("Change in mean June\u2013August monthly SPEI-3, "
                          f"{start_year}\u2013{end_year} "
                          "(positive values indicate drying; SPEI units decade$^{-1}$)")
    norm = BoundaryNorm(boundaries, ncolors=cmap.N, clip=True)

    with mpl.rc_context({"font.size": pt(8), "axes.labelsize": pt(8.5),
                         "xtick.labelsize": pt(8), "ytick.labelsize": pt(8),
                         "axes.linewidth": 0.4 * FIG_SCALE,
                         "pdf.fonttype": 42, "ps.fonttype": 42}):

        fig = plt.figure(figsize=(dl.AGU_MAX_WIDTH_IN * FIG_SCALE, 5.55 * FIG_SCALE),
                         facecolor="white")
        grid = fig.add_gridspec(2, 1, height_ratios=[0.74, 1.26], hspace=0.20)
        ax_time = fig.add_subplot(grid[0])
        ax_map = fig.add_subplot(grid[1])

        years = national["Year"].to_numpy()
        series = national["JJA_Mean_SPEI3"].to_numpy()

        ax_time.bar(years, series, width=0.86,
                    color=np.where(series < 0, dl.DRY_COLOR, dl.WET_COLOR),
                    edgecolor="white", linewidth=0.1 * FIG_SCALE, alpha=0.94, zorder=2)
        ax_time.axhline(0, color="black", linewidth=0.36 * FIG_SCALE, zorder=3)
        ax_time.axhline(dl.DRY_THRESHOLD, color=dl.DRY_COLOR, linestyle="--",
                        linewidth=0.6 * FIG_SCALE, zorder=3)
        ax_time.plot(national["Year"], national["OLS_fitted"], color=TREND_COLOR,
                     linewidth=1.0 * FIG_SCALE, zorder=5)

        p_text = ("$p<0.001$" if national_fit.pvalue < 0.001
                  else rf"$p={national_fit.pvalue:.3f}$")
        ax_time.text(0.985, 0.95,
                     f"OLS trend: {national_fit.slope * 10:+.3f} "
                     + "SPEI units decade$^{-1}$\n"
                     + rf"$r={national_fit.rvalue:.2f}$, {p_text}",
                     transform=ax_time.transAxes, ha="right", va="top",
                     fontsize=pt(8), color=TREND_COLOR,
                     bbox={"facecolor": "white", "edgecolor": "none",
                           "alpha": 0.90, "pad": 5}, zorder=7)

        ax_time.text(0.012, dl.DRY_THRESHOLD + 0.04,
                     f"Dry-summer threshold\n(SPEI-3 = {dl.DRY_THRESHOLD:.1f})",
                     transform=ax_time.get_yaxis_transform(), ha="left", va="bottom",
                     fontsize=pt(8), color=dl.DRY_COLOR,
                     bbox={"facecolor": "white", "edgecolor": "none",
                           "alpha": 0.88, "pad": 4}, zorder=7)

        ax_time.set_xlim(start_year - 1, end_year + 1)
        ax_time.set_xticks(np.arange(start_year, end_year + 1, tick_step))
        ax_time.set_xlabel("Year", fontsize=pt(8.5), labelpad=14)
        ax_time.set_ylabel("Mean June\u2013August\nmonthly SPEI-3",
                           fontsize=pt(8.5), labelpad=18)
        ax_time.tick_params(axis="both", which="major", labelsize=pt(8),
                            direction="out", length=2.2 * FIG_SCALE,
                            width=0.36 * FIG_SCALE)
        ax_time.grid(axis="y", linestyle="--", linewidth=0.3 * FIG_SCALE, alpha=0.30)
        ax_time.grid(axis="x", visible=False)
        ax_time.spines[["top", "right"]].set_visible(False)

        ax_map.set_axisbelow(True)
        mapped.plot(column="Drying_rate_per_decade", cmap=cmap, norm=norm,
                    linewidth=0.33 * FIG_SCALE, edgecolor="#606060", ax=ax_map,
                    legend=False, zorder=2)

        dl.draw_aimag_labels(ax_map, mapped, fontsize=pt(8), scale=FIG_SCALE,
                             offsets=LABEL_OFFSETS)

        min_lon, min_lat, max_lon, max_lat = mapped.total_bounds
        dl.expand_limits_for_labels(ax_map, mapped)
        ax_map.set_xticks(np.arange(88, 121, 4))
        ax_map.set_yticks(np.arange(42, 53, 2))
        ax_map.xaxis.set_major_formatter(
            FuncFormatter(lambda v, p: f"{v:.0f}\u00b0E"))
        ax_map.yaxis.set_major_formatter(
            FuncFormatter(lambda v, p: f"{v:.0f}\u00b0N"))
        ax_map.set_xlabel("Longitude", fontsize=pt(8.5), labelpad=14)
        ax_map.set_ylabel("Latitude", fontsize=pt(8.5), labelpad=14)
        ax_map.tick_params(axis="both", which="major", labelsize=pt(8),
                           direction="out", length=2.2 * FIG_SCALE,
                           width=0.36 * FIG_SCALE)
        ax_map.grid(color="#777777", linestyle=":", linewidth=0.28 * FIG_SCALE,
                    alpha=0.40, zorder=0)
        ax_map.set_aspect(1 / np.cos(np.deg2rad((min_lat + max_lat) / 2)))

        for spine in ax_map.spines.values():
            spine.set_visible(True)
            spine.set_color("#555555")
            spine.set_linewidth(0.33 * FIG_SCALE)

        scalar = ScalarMappable(norm=norm, cmap=cmap)
        scalar.set_array([])
        colorbar = fig.colorbar(scalar, ax=ax_map, orientation="horizontal",
                                boundaries=boundaries, ticks=boundaries,
                                spacing="uniform", fraction=0.060, pad=0.16,
                                shrink=0.78, aspect=32)
        colorbar.set_label(colorbar_label, fontsize=pt(8.5), labelpad=16)
        colorbar.ax.set_xticklabels([f"{v:.2f}" for v in boundaries])
        colorbar.ax.tick_params(axis="x", which="major", labelsize=pt(8),
                                length=2.2 * FIG_SCALE, width=0.3 * FIG_SCALE, pad=8)

        ax_time.text(-0.065, 1.02, "a", transform=ax_time.transAxes,
                     fontsize=pt(10), fontweight="bold", ha="left", va="bottom")
        ax_map.text(-0.065, 1.02, "b", transform=ax_map.transAxes,
                    fontsize=pt(10), fontweight="bold", ha="left", va="bottom")

        fig.subplots_adjust(left=0.105, right=0.98, top=0.975, bottom=0.09)

    return fig, national, national_fit, trends

## Figure 5. June-August SPEI-3 trends, 1992-2024

In [ ]:
fig_5, national_1992, fit_1992, trends_1992 = build_spei_trend_figure(
    dl.MORT_START, dl.MORT_END, tick_step=5)
dl.save_figure(fig_5, "figure_05_spei3_trends_1992_2024", scale=FIG_SCALE)
plt.show()

print(f"National area-weighted JJA SPEI-3 trend, {dl.MORT_START}-{dl.MORT_END}")
print(f"  slope {fit_1992.slope * 10:+.4f} SPEI units per decade")
print(f"  r = {fit_1992.rvalue:.3f}, p = {fit_1992.pvalue:.3g}, "
      f"n = {len(national_1992)}")
print(f"  years below the dry-summer threshold: "
      f"{(national_1992['JJA_Mean_SPEI3'] < dl.DRY_THRESHOLD).sum()} of "
      f"{len(national_1992)}")

## Figure S1. June-August SPEI-3 trends, 1950-2024

In [ ]:
fig_s1, national_1950, fit_1950, trends_1950 = build_spei_trend_figure(
    dl.SPEI_START, dl.SPEI_END, tick_step=10)
dl.save_figure(fig_s1, "figure_S01_spei3_trends_1950_2024", scale=FIG_SCALE)
plt.show()

print(f"National area-weighted JJA SPEI-3 trend, {dl.SPEI_START}-{dl.SPEI_END}")
print(f"  slope {fit_1950.slope * 10:+.4f} SPEI units per decade")
print(f"  r = {fit_1950.rvalue:.3f}, p = {fit_1950.pvalue:.3g}, "
      f"n = {len(national_1950)}")

## Aimag drying rates

Both analysis periods are printed side by side. The manuscript reports the 1992-2024
ranking in section 3.2 and the 1950-2024 ranking in Supplementary Table S6, so the
two orderings are not expected to agree.

In [ ]:
for period, group in spei_trends.groupby("Period"):
    print(f"\n{period}, ranked from fastest to slowest drying")
    print(group[["Aimag", "Drying_rate_per_decade", "SPEI_slope_per_decade",
                 "r", "p_value", "N_years"]]
          .round(4).to_string(index=False))
    significant = (group["p_value"] < 0.05).sum()
    print(f"  significant at p < 0.05: {significant} of {len(group)} aimags")

In [ ]:
wide = (spei_trends.pivot(index="Aimag", columns="Period",
                          values="Drying_rate_per_decade")
        .round(3))
wide["Rank_1992_2024"] = wide[f"{dl.MORT_START}-{dl.MORT_END}"].rank(
    ascending=False).astype(int)
wide["Rank_1950_2024"] = wide[f"{dl.SPEI_START}-{dl.SPEI_END}"].rank(
    ascending=False).astype(int)
wide["Rank_shift"] = wide["Rank_1950_2024"] - wide["Rank_1992_2024"]

print("Drying rate per decade by period, with ranking shift")
print(wide.sort_values("Rank_1992_2024").to_string())